# Contextual Compression Retriever — 맥락 기반 문서 압축 검색기

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

- **ContextualCompressionRetriever**의 작동 원리 이해하기
- `LLMChainExtractor`, `LLMChainFilter`, `EmbeddingsFilter` 각각의 차이 파악하기
- `DocumentCompressorPipeline`으로 여러 압축 단계를 조합하기

## 사전 지식

- 01-VectorStoreRetriever에서 기본 Retriever를 만들어본 경험
- LLM(대규모 언어 모델)과 임베딩 모델의 역할 이해

---

```mermaid
flowchart LR
    Q[사용자 쿼리]:::input --> BR[Base Retriever<br/>후보 문서 검색]:::process
    BR --> C{Compressor<br/>선택}:::process
    C -- LLMExtractor --> E1[LLM이<br/>관련 내용 추출]:::process
    C -- LLMFilter --> E2[LLM이<br/>문서 선별]:::process
    C -- EmbeddingsFilter --> E3[임베딩<br/>유사도 필터]:::process
    E1 --> O[압축된<br/>문서]:::output
    E2 --> O
    E3 --> O

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
```

## 핵심 문제

일반 Retriever는 문서 전체를 반환하지만, 실제로 필요한 정보는 문서의 일부분일 수 있어요:

- **문제 1**: 무관한 텍스트가 많이 포함되어 LLM 비용 증가
- **문제 2**: 노이즈가 많아 응답 품질 저하
- **문제 3**: 컨텍스트 길이 제한으로 중요 정보 누락

## 해결 방법

`ContextualCompressionRetriever`는 두 단계로 작동해요:

1. **검색(Retrieval)**: Base Retriever로 후보 문서 검색
2. **압축(Compression)**: Document Compressor로 관련 내용만 추출

## 환경 설정

In [1]:
from dotenv import load_dotenv

# API 키 정보 로드
load_dotenv()

True

In [3]:

from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import (
  DocumentCompressorPipeline,
  EmbeddingsFilter,
  LLMChainExtractor,
  LLMChainFilter
)



## 1. 기본 Retriever 준비

먼저 압축하지 않은 기본 retriever의 결과를 확인해봅시다.

In [4]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ---------------------------------------------------
# 압축 기법 비교를 위한 기본 Retriever 준비
# ---------------------------------------------------

# ============================================================
# TODO: ai-story.txt를 로드하고 800자 청크로 분할한 뒤 FAISS Retriever를 생성하세요
# 힌트: chunk_size=800으로 크게 설정해야 압축 효과가 잘 보임
# 예상 결과: 총 청크 수와 기본 Retriever 생성 완료 메시지 출력
# ============================================================

# 문서 로드 및 분할
loader = TextLoader("./data/ai-story.txt")
documents = loader.load()

# 큰 청크를 사용해야 압축 후 크기 감소 효과가 뚜렷이 보임
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)
split_docs = text_splitter.split_documents(documents)


# 벡터 데이터베이스 생성
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(split_docs, embeddings)

# 기본 retriever 생성
base_retriever = vectorstore.as_retriever()


In [5]:
# 헬퍼 함수: 문서를 깔끔하게 출력
def print_docs(docs, title="검색 결과"):
    print(f"\n{'='*80}")
    print(f"{title}")
    print(f"{'='*80}\n")
    
    for i, doc in enumerate(docs, 1):
        print(f"[문서 {i}]")
        print(f"내용: {doc.page_content}")
        print(f"길이: {len(doc.page_content)} 문자")
        print("-" * 80 + "\n")


In [7]:
# ---------------------------------------------------
# 기본 Retriever 결과를 확인해 압축 전 상태 파악
# ---------------------------------------------------

# ============================================================
# TODO: base_retriever로 쿼리를 검색하고 총 문자 수를 계산하세요
# 힌트: retriever.invoke(query) 후 sum(len(d.page_content) for d in docs)
# 예상 결과: 원본 크기 확인 (압축 후와 비교할 기준선)
# ============================================================

# 테스트 쿼리
query = "Hugging Face의 Transformers 라이브러리는 어떤 작업을 지원하나요?"

# 기본 retriever로 검색
docs = base_retriever.invoke(query)


# 전체 문자 수 계산
print(f"쿼리 {query}")
print_docs(docs, "기본 retriever 결과(무압축)")

total_chars = sum(len(d.page_content) for d in docs)
print(f"압축 전 전체 결과 문자 수: {total_chars}자")


쿼리 Hugging Face의 Transformers 라이브러리는 어떤 작업을 지원하나요?

기본 retriever 결과(무압축)

[문서 1]
내용: HuggingFace

Hugging Face는 자연어 처리(NLP) 분야에서 선도적인 역할을 하는 기술 회사로, 인공 지능(AI) 연구와 응용 프로그램 개발을 위한 다양한 오픈 소스 도구와 라이브러리를 제공합니다. 2016년에 설립된 이 회사는 특히 'Transformers' 라이브러리를 통해 큰 인기를 얻었습니다. Transformers 라이브러리는 BERT, GPT, RoBERTa, T5와 같은 최신의 변환기(Transformer) 기반 모델을 쉽게 사용할 수 있게 해주며, Python 프로그래밍 언어로 작성되어 있습니다. 이 라이브러리의 주요 목적은 최신의 NLP 기술을 쉽고 효율적으로 접근할 수 있도록 하는 것입니다.

Hugging Face의 Transformers 라이브러리는 다양한 NLP 작업에 활용될 수 있습니다. 이에는 텍스트 분류, 정보 추출, 질문 응답, 요약 생성, 텍스트 생성 등이 포함되며, 다양한 언어에 대한 지원도 제공합니다. 라이브러리는 사전 훈련된 모델을 포함하고 있어, 사용자가 복잡한 모델을 처음부터 훈련시키지 않고도, 빠르게 고성능의 NLP 시스템을 구축할 수 있게 합니다.

Hugging Face는 또한 'Model Hub'를 운영하고 있습니다. 이는 수천 개의 사전 훈련된 모델과 데이터셋을 호스팅하는 플랫폼으로, 연구자와 개발자가 자신의 모델을 공유하고 다른 사람들의 모델을 쉽게 찾아 사용할 수 있게 합니다. Model Hub를 통해 사용자는 특정 NLP 작업에 최적화된 모델을 검색하고, 직접 훈련시킨 모델을 커뮤니티와 공유할 수 있습니다.
길이: 783 문자
--------------------------------------------------------------------------------

[문서 2]
내용: Hugging Face는 또한 AI 연구 커뮤니티

## 2. LLMChainExtractor - LLM 기반 내용 추출

`LLMChainExtractor`는 LLM을 사용하여 쿼리와 관련된 내용만 추출합니다.

### 작동 방식

1. 검색된 각 문서에 대해 LLM에게 질문
2. "이 문서에서 쿼리와 관련된 부분만 추출하세요"
3. LLM이 관련 내용만 반환

### 장점과 단점

**장점:**
- 매우 정확한 내용 추출
- 문맥을 이해하여 관련성 판단

**단점:**
- LLM 호출 비용 발생
- 속도가 느림 (각 문서마다 LLM 호출)

In [8]:
from langchain_openai import ChatOpenAI

# LLM 초기화
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# LLMChainExtractor 생성
compressor = LLMChainExtractor.from_llm(llm)

# ContextualCompressionRetriever 생성
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)



In [16]:
# 압축된 검색 실행
compressed_docs = compression_retriever.invoke(query)

# 압축 효과 비교
print(f":memo: 쿼리: {query}<br/>")
print_docs(compressed_docs, "LLMChainExtractor 적용 후")

# 압축 효과 비교
original_total = sum(len(doc.page_content) for doc in docs)
compressed_total = sum(len(doc.page_content) for doc in compressed_docs)
reduction = (1 - compressed_total / original_total) * 100

print(f"<br/>:bar_chart: 압축 효과:")
print(f"  원본: {original_total}자")
print(f"  압축 후: {compressed_total}자")
print(f"  감소율: {reduction:.1f}%")
print(f"<br/>:bulb: 장점: 쿼리와 직접 관련된 내용만 추출되어 품질 향상")

# 아래에 코드를 작성하세요


:memo: 쿼리: Hugging Face의 Transformers 라이브러리는 어떤 작업을 지원하나요?<br/>

LLMChainExtractor 적용 후

[문서 1]
내용: Hugging Face의 Transformers 라이브러리는 다양한 NLP 작업에 활용될 수 있습니다. 이에는 텍스트 분류, 정보 추출, 질문 응답, 요약 생성, 텍스트 생성 등이 포함되며, 다양한 언어에 대한 지원도 제공합니다. 라이브러리는 사전 훈련된 모델을 포함하고 있어, 사용자가 복잡한 모델을 처음부터 훈련시키지 않고도, 빠르게 고성능의 NLP 시스템을 구축할 수 있게 합니다.
길이: 216 문자
--------------------------------------------------------------------------------

[문서 2]
내용: 변환기 모델의 이러한 혁신적인 접근 방식은 NLP를 비롯한 여러 분야에서 큰 변화를 가져왔습니다. BERT, GPT, RoBERTa 등의 후속 모델들은 모두 "Attention Is All You Need" 논문에서 제시된 변환기 아키텍처를 기반으로 하며, 이들 모델은 텍스트 분류, 요약, 번역, 질문 응답 시스템 등 다양한 작업에서 인상적인 성능을 보여줍니다.
길이: 203 문자
--------------------------------------------------------------------------------

<br/>:bar_chart: 압축 효과:
  원본: 2908자
  압축 후: 419자
  감소율: 85.6%
<br/>:bulb: 장점: 쿼리와 직접 관련된 내용만 추출되어 품질 향상


## 3. LLMChainFilter - LLM 기반 문서 필터링

`LLMChainFilter`는 문서 내용을 압축하지 않고, 관련 있는 문서만 선택적으로 반환합니다.

### LLMChainExtractor vs LLMChainFilter

| 특징 | LLMChainExtractor | LLMChainFilter |
|------|------------------|----------------|
| **동작** | 문서 내용 추출/압축 | 문서 전체 유지 또는 제거 |
| **결과** | 압축된 내용 반환 | 원본 문서 또는 빈 결과 |
| **용도** | 긴 문서에서 핵심 추출 | 관련 없는 문서 제거 |
| **속도** | 느림 | 느림 |
| **비용** | 높음 (추출 작업) | 높음 (판단 작업) |

In [9]:
# ---------------------------------------------------
# LLMChainFilter로 관련 없는 문서 제거 (내용 압축 없이)
# ---------------------------------------------------

# ============================================================
# TODO: LLMChainFilter를 사용해 관련 없는 문서를 제거하는 retriever를 만들고 검색하세요
# 힌트: LLMChainFilter.from_llm(llm) → ContextualCompressionRetriever
# 예상 결과: 원본 문서를 그대로 유지하되 관련 없는 문서는 제거됨
# ============================================================

# LLMChainFilter 생성
# LLMChainFilter: 문서 내용을 압축하지 않고 관련/비관련 이진 판단
docs_filter = LLMChainFilter.from_llm(llm)

# ContextualCompressionRetriever 생성
filter_retriever = ContextualCompressionRetriever(
    base_compressor=docs_filter,
    base_retriever=base_retriever
)

# 필터링된 검색 실행

filtered_docs = filter_retriever.invoke(query)

print(f"쿼리: {query}")
print_docs(filtered_docs, "LLMChainFilter 적용 후")


쿼리: Hugging Face의 Transformers 라이브러리는 어떤 작업을 지원하나요?

LLMChainFilter 적용 후

[문서 1]
내용: HuggingFace

Hugging Face는 자연어 처리(NLP) 분야에서 선도적인 역할을 하는 기술 회사로, 인공 지능(AI) 연구와 응용 프로그램 개발을 위한 다양한 오픈 소스 도구와 라이브러리를 제공합니다. 2016년에 설립된 이 회사는 특히 'Transformers' 라이브러리를 통해 큰 인기를 얻었습니다. Transformers 라이브러리는 BERT, GPT, RoBERTa, T5와 같은 최신의 변환기(Transformer) 기반 모델을 쉽게 사용할 수 있게 해주며, Python 프로그래밍 언어로 작성되어 있습니다. 이 라이브러리의 주요 목적은 최신의 NLP 기술을 쉽고 효율적으로 접근할 수 있도록 하는 것입니다.

Hugging Face의 Transformers 라이브러리는 다양한 NLP 작업에 활용될 수 있습니다. 이에는 텍스트 분류, 정보 추출, 질문 응답, 요약 생성, 텍스트 생성 등이 포함되며, 다양한 언어에 대한 지원도 제공합니다. 라이브러리는 사전 훈련된 모델을 포함하고 있어, 사용자가 복잡한 모델을 처음부터 훈련시키지 않고도, 빠르게 고성능의 NLP 시스템을 구축할 수 있게 합니다.

Hugging Face는 또한 'Model Hub'를 운영하고 있습니다. 이는 수천 개의 사전 훈련된 모델과 데이터셋을 호스팅하는 플랫폼으로, 연구자와 개발자가 자신의 모델을 공유하고 다른 사람들의 모델을 쉽게 찾아 사용할 수 있게 합니다. Model Hub를 통해 사용자는 특정 NLP 작업에 최적화된 모델을 검색하고, 직접 훈련시킨 모델을 커뮤니티와 공유할 수 있습니다.
길이: 783 문자
--------------------------------------------------------------------------------

[문서 2]
내용: 어텐션 메커니즘은 입력 데이터의 서로 다른 부분에

## 4. EmbeddingsFilter - 임베딩 기반 필터링

`EmbeddingsFilter`는 임베딩 유사도를 사용하여 빠르고 저렴하게 문서를 필터링합니다.

### 장점

- **빠른 속도**: LLM 호출 없이 벡터 연산만 수행
- **낮은 비용**: 임베딩 유사도 계산만 필요
- **효율성**: 대량의 문서 필터링에 적합

### 언제 사용하면 좋을까요?

- 빠른 응답이 필요한 애플리케이션
- 비용을 최소화해야 할 때
- 이미 검색된 문서를 추가로 필터링할 때

> **실무 팁**: 프로덕션 환경에서 Compression Retriever가 필요하다면 EmbeddingsFilter를 먼저 선택하세요. LLM 호출 없이 동작하므로 레이턴시 증가가 거의 없고, 비용도 절감됩니다.

In [10]:
# EmbeddingsFilter 생성 (유사도 임계값 0.75)
embeddings_filter = EmbeddingsFilter(
    embeddings=embeddings,
    similarity_threshold=0.472
)
# ContextualCompressionRetriever 생성
embeddings_filter_retriever = ContextualCompressionRetriever(
    base_compressor=embeddings_filter,
    base_retriever=base_retriever
)
# 임베딩 기반 필터링 실행
embedding_filtered_docs = embeddings_filter_retriever.invoke(query)

# 아래에 코드를 작성하세요
print(f":memo: 쿼리: {query}<br/>")
print_docs(embedding_filtered_docs, "EmbeddingsFilter 적용 후")


:memo: 쿼리: Hugging Face의 Transformers 라이브러리는 어떤 작업을 지원하나요?<br/>

EmbeddingsFilter 적용 후

[문서 1]
내용: HuggingFace

Hugging Face는 자연어 처리(NLP) 분야에서 선도적인 역할을 하는 기술 회사로, 인공 지능(AI) 연구와 응용 프로그램 개발을 위한 다양한 오픈 소스 도구와 라이브러리를 제공합니다. 2016년에 설립된 이 회사는 특히 'Transformers' 라이브러리를 통해 큰 인기를 얻었습니다. Transformers 라이브러리는 BERT, GPT, RoBERTa, T5와 같은 최신의 변환기(Transformer) 기반 모델을 쉽게 사용할 수 있게 해주며, Python 프로그래밍 언어로 작성되어 있습니다. 이 라이브러리의 주요 목적은 최신의 NLP 기술을 쉽고 효율적으로 접근할 수 있도록 하는 것입니다.

Hugging Face의 Transformers 라이브러리는 다양한 NLP 작업에 활용될 수 있습니다. 이에는 텍스트 분류, 정보 추출, 질문 응답, 요약 생성, 텍스트 생성 등이 포함되며, 다양한 언어에 대한 지원도 제공합니다. 라이브러리는 사전 훈련된 모델을 포함하고 있어, 사용자가 복잡한 모델을 처음부터 훈련시키지 않고도, 빠르게 고성능의 NLP 시스템을 구축할 수 있게 합니다.

Hugging Face는 또한 'Model Hub'를 운영하고 있습니다. 이는 수천 개의 사전 훈련된 모델과 데이터셋을 호스팅하는 플랫폼으로, 연구자와 개발자가 자신의 모델을 공유하고 다른 사람들의 모델을 쉽게 찾아 사용할 수 있게 합니다. Model Hub를 통해 사용자는 특정 NLP 작업에 최적화된 모델을 검색하고, 직접 훈련시킨 모델을 커뮤니티와 공유할 수 있습니다.
길이: 783 문자
--------------------------------------------------------------------------------



## 5. DocumentCompressorPipeline - 여러 Compressor 조합

`DocumentCompressorPipeline`을 사용하면 여러 compressor와 transformer를 순차적으로 적용할 수 있습니다.

### 파이프라인 구성 요소

1. **TextSplitter**: 문서를 더 작은 청크로 분할
2. **EmbeddingsRedundantFilter**: 중복 문서 제거 (유사도 0.95 이상)
3. **EmbeddingsFilter**: 관련성 필터링 (유사도 임계값)
4. **LLMChainExtractor**: 최종 내용 추출

### 파이프라인 전략

```
큰 문서 → 작은 청크 → 중복 제거 → 관련성 필터링 → 내용 추출
```

> **실무 팁**: 파이프라인 단계 순서가 중요합니다. 비용이 낮은 단계(TextSplitter, EmbeddingsFilter)를 먼저 적용해 문서를 줄이고, 비용이 높은 단계(LLMChainExtractor)는 마지막에 적용하세요.

In [13]:
from langchain_community.document_transformers import EmbeddingsRedundantFilter
from langchain_text_splitters import CharacterTextSplitter

# 1. 텍스트 분할기 (문서를 더 작은 청크로)
splitter = CharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50
)

# 2. 중복 필터 (유사도 0.95 이상인 문서 제거)
redundant_filter = EmbeddingsRedundantFilter(embeddings=embeddings)

# 3. 관련성 필터 (유사도 0.75 이상만 유지)
relevant_filter = EmbeddingsFilter(
    embeddings=embeddings,
    similarity_threshold=0.472
)

# 4. LLM 내용 추출기
extractor = LLMChainExtractor.from_llm(llm)

# 파이프라인 생성
pipeline_compressor = DocumentCompressorPipeline(
    transformers=[splitter, redundant_filter, relevant_filter, extractor]
)

# 아래에 코드를 작성하세요


In [17]:
# ---------------------------------------------------
# 파이프라인 적용 후 모든 방식 비교
# ---------------------------------------------------

# ============================================================
# TODO: pipeline_retriever로 검색하고 4가지 방식의 문서 수와 총 문자 수를 비교하세요
# 힌트: pipeline_retriever.invoke(query) 후 비교표 출력
# 예상 결과: Pipeline이 가장 압축된 결과를 반환
# ============================================================

# 파이프라인 적용
pipeline_retriever = ContextualCompressionRetriever(
    base_compressor=pipeline_compressor,
    base_retriever=base_retriever
)

# 파이프라인 실행
pipeline_docs = pipeline_retriever.invoke(query)
print(f":memo: 쿼리: {query}<br/>")
print_docs(pipeline_docs, "DocumentCompressorPipeline 적용 후")

# 전체 비교
print(f"<br/>:bar_chart: 압축 효과 비교:")
print(f"{'방식':<30} {'문서수':<10} {'총 문자수':<15}")
print("-" * 60)
print(f"{'기본 Retriever':<30} {len(docs):<10} {sum(len(d.page_content) for d in docs):<15}")
print(f"{'LLMChainExtractor':<30} {len(compressed_docs):<10} {sum(len(d.page_content) for d in compressed_docs):<15}")
print(f"{'EmbeddingsFilter':<30} {len(embedding_filtered_docs):<10} {sum(len(d.page_content) for d in embedding_filtered_docs):<15}")
print(f"{'Pipeline (all combined)':<30} {len(pipeline_docs):<10} {sum(len(d.page_content) for d in pipeline_docs):<15}")

:memo: 쿼리: Hugging Face의 Transformers 라이브러리는 어떤 작업을 지원하나요?<br/>

DocumentCompressorPipeline 적용 후

[문서 1]
내용: Hugging Face의 Transformers 라이브러리는 다양한 NLP 작업에 활용될 수 있습니다. 이에는 텍스트 분류, 정보 추출, 질문 응답, 요약 생성, 텍스트 생성 등이 포함되며, 다양한 언어에 대한 지원도 제공합니다.
길이: 128 문자
--------------------------------------------------------------------------------

[문서 2]
내용: Transformers 라이브러리는 BERT, GPT, RoBERTa, T5와 같은 최신의 변환기(Transformer) 기반 모델을 쉽게 사용할 수 있게 해주며, Python 프로그래밍 언어로 작성되어 있습니다. 이 라이브러리의 주요 목적은 최신의 NLP 기술을 쉽고 효율적으로 접근할 수 있도록 하는 것입니다.
길이: 175 문자
--------------------------------------------------------------------------------

<br/>:bar_chart: 압축 효과 비교:
방식                             문서수        총 문자수          
------------------------------------------------------------
기본 Retriever                   4          2908           
LLMChainExtractor              2          419            
EmbeddingsFilter               1          783            
Pipeline (all combined)        2          303            


## 6. Compressor 선택 가이드

### 사용 시나리오별 추천

| 시나리오 | 추천 Compressor | 이유 |
|---------|----------------|------|
| **최고 품질 필요** | LLMChainExtractor | 가장 정확한 내용 추출 |
| **빠른 응답 필요** | EmbeddingsFilter | LLM 호출 없음 |
| **비용 최소화** | EmbeddingsFilter | 임베딩만 사용 |
| **문서 선택만** | LLMChainFilter | 원본 유지하며 필터링 |
| **최적 균형** | Pipeline | 단계별 최적화 |

### 비용 vs 품질 트레이드오프

```
높은 품질 ← → 낮은 비용/빠른 속도

LLMChainExtractor > Pipeline > LLMChainFilter > EmbeddingsFilter
```

### 실전 권장 조합

```python
# Production 환경 권장
pipeline = DocumentCompressorPipeline([
    CharacterTextSplitter(chunk_size=400),  # 작은 청크
    EmbeddingsRedundantFilter(),           # 중복 제거
    EmbeddingsFilter(threshold=0.75),      # 빠른 필터링
    # LLMChainExtractor()  # 선택적: 최종 품질 향상
])
```

## 마무리 정리

이 노트북에서 배운 내용을 정리해요:

| Compressor | 비용 | 속도 | 품질 | 추천 상황 |
|-----------|------|------|------|----------|
| **LLMChainExtractor** | 높음 | 느림 | 최고 | 품질이 최우선 |
| **LLMChainFilter** | 높음 | 느림 | 높음 | 원본 유지하며 필터링 |
| **EmbeddingsFilter** | 낮음 | 빠름 | 보통 | 비용·속도 중시 |
| **Pipeline** | 중간 | 중간 | 높음 | 균형 잡힌 운영 |

### 다음 노트북 예고

**03-EnsembleRetriever**: BM25 키워드 검색과 벡터 검색을 결합하는 하이브리드 검색을 배워요. 검색 방식의 약점을 서로 보완하는 실무 필수 기법입니다.